# 🛒 Retail Sales & Customer Analysis
### Exploratory Data Analysis (EDA)

**Author:** Jatindra Patel  
**Tools Used:** Python, Pandas, NumPy, Matplotlib, Plotly  
**Environment:** Google Colab

## 📌 Project Description

This project performs Exploratory Data Analysis (EDA) on a retail dataset to extract insights related to sales performance, customer behavior, and operational efficiency.

The goal is to support data-driven decision-making for business growth.

## 🎯 Business Objectives

- Identify top-performing products and categories  
- Analyze sales trends over time  
- Find high-value customers  
- Evaluate discount effectiveness  
- Analyze shipping performance  
- Understand customer purchasing behavior  

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt

In [5]:
fact_sales = pd.read_csv('/content/Fact_Sales.csv')
orders = pd.read_csv('/content/orders.csv')
products = pd.read_csv('/content/products.csv', encoding='latin1')
shippers = pd.read_csv('/content/shippers.csv')
categories = pd.read_csv('/content/categories.csv')
customers = pd.read_csv('/content/customers.csv', encoding='latin1')
order_details = pd.read_csv('/content/order_details.csv')

## 📊 Data Understanding

This dataset follows a star schema:

- Fact Table: Fact_Sales  
- Dimension Tables: Orders, Products, Shippers, Order_details,      Customers, Categories

In [13]:
fact_sales.head()
orders.head()

,OrderID,CustomerID,OrderDate,RequiredDate,ShippedDate,ShipperID,Freight
0,10248,VINET,2013-07-04,8/1/2013,7/16/2013,3,32.38
1,10249,TOMSP,2013-07-05,8/16/2013,7/10/2013,1,11.61
2,10250,HANAR,2013-07-08,8/5/2013,7/12/2013,2,65.83
3,10251,VICTE,2013-07-08,8/5/2013,7/15/2013,1,41.34
4,10252,SUPRD,2013-07-09,8/6/2013,7/11/2013,2,51.30


## 🧹 Data Cleaning & Preparation

In [12]:
orders['OrderDate'] = pd.to_datetime(orders['OrderDate'])

sales = fact_sales.merge(orders, on='OrderID', how='left')
sales = sales.merge(products, on='ProductID', how='left')

sales.isnull().sum()

,0
OrderID,0
ProductID,0
UnitPrice_x,0
Quantity,0
Discount,0
CustomerID_x,0
OrderDate_x,0
ShipperID_x,0
Freight_x,0
SalesAmount,0


In [11]:
sales['ShippingStatus'] = np.where(
    sales['ShippedDate'].isnull(),
    'Not Shipped',
    'Shipped'
)

## 📈 Analysis 1: Monthly Sales Trend

### Business Question
How does sales change over time?

### Insight
This helps identify growth trends and seasonal patterns.

In [15]:
sales['Month'] = sales['OrderDate_y'].dt.to_period('M').astype(str)

monthly_sales = sales.groupby('Month')['SalesAmount'].sum().reset_index()

fig = px.line(
    monthly_sales,
    x='Month',
    y='SalesAmount',
    markers=True,
    title='Monthly Sales Trend'
)

fig.show()

# 🏆 Analysis 2: Top 10 Best Selling Products

**Business Question**

Which products generate the highest revenue?

**Insight**

Identifies top revenue-driving products.

In [23]:
top_products = sales.groupby('ProductName')['SalesAmount'].sum().reset_index()

top_products = top_products.sort_values(
    by='SalesAmount',
    ascending=False
).head(10)

fig = px.bar(
    top_products,
    x='SalesAmount',
    y='ProductName',
    orientation='h',
    title='Top 10 Revenue Generating Products'
)

fig.show()

#💰 Analysis 3: Order Value Distribution

**Business Question**

What is the distribution of order values?

**Insight**

Reveals customer spending patterns.

In [51]:
order_value = sales.groupby('OrderID')['SalesAmount'].sum().reset_index()

fig = px.histogram(
    order_value,
    x='SalesAmount',
    nbins=40,
    title='Order Value Distribution'
)

fig.show()

#🎯 Analysis 4: Discount vs Sales

**Business Question**

How do discounts impact sales?

**Insight**

Shows relationship between discounts and sales.

In [26]:
fig = px.scatter(
    sales,
    x='Discount',
    y='SalesAmount',
    title='Discount vs Sales Impact'
)

fig.show()

#👑 Analysis 5: Top Customers by Revenue

**Business Question**

Who are the top customers?

**Insight**

Identifies high-value customers.

In [52]:
customer_sales = sales.groupby('CustomerID_y')['SalesAmount'].sum().reset_index()

top_customers = customer_sales.sort_values(
    by='SalesAmount',
    ascending=False
).head(10)

fig = px.bar(
    top_customers,
    x='CustomerID_y',
    y='SalesAmount',

    title='Top Customers by Revenue'
)

fig.update_layout(xaxis_tickangle=270)

fig.show()

#🚚 Analysis 6: Shipping Company Performance

**Business Question**

Which shippers handle most orders?

**Insight**

Highlights top logistics partners.

In [54]:
ship_data = orders.merge(shippers, left_on='ShipperID', right_on='shipperID')

ship_orders = ship_data.groupby('companyName')['OrderID'].count().reset_index()

fig = px.bar(
    ship_orders,
    x='companyName',
    y='OrderID',
    text='OrderID',
    title='Orders Handled by Shipping Companies'
)
fig.update_traces(textposition='outside')
fig.show()

#📉 Analysis 7: Pareto Analysis

**Business Question**

Do few products drive most revenue?

**Insight**

Shows revenue concentration (80/20 rule).

In [56]:
product_sales = sales.groupby('ProductName')['SalesAmount'].sum().reset_index()

product_sales = product_sales.sort_values(
    by='SalesAmount',
    ascending=False
)

product_sales['CumulativeSales'] = product_sales['SalesAmount'].cumsum()

product_sales['CumulativePercent'] = (
    product_sales['CumulativeSales'] /
    product_sales['SalesAmount'].sum()
)

fig = px.line(
    product_sales,
    y='CumulativePercent',
    title='Pareto Analysis of Product Sales'
)

fig.show()

#🔥 RFM Customer Segmentation

**Business Question**

Which customers are most valuable, loyal, or at risk of churning?

**Insight**

Customers with:

- Low Recency + High Frequency + High Monetary → Loyal Customers

- High Recency → At Risk Customers

In [59]:
import pandas as pd
from datetime import datetime

snapshot_date = sales['OrderDate_y'].max() + pd.Timedelta(days=1)
rfm = sales.groupby('CustomerID_y').agg({
    'OrderDate_y': lambda x: (snapshot_date - x.max()).days,
    'OrderID': 'nunique',
    'SalesAmount': 'sum'
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']




In [62]:
import plotly.express as px

fig = px.scatter(
    rfm,
    x='Recency',
    y='Monetary',
    size='Frequency',

    title='🧠 RFM Customer Segmentation',

)

fig.show()

#💰 Customer Lifetime Value (CLV)

**Business Question**

Which customers will generate the highest long-term revenue?

**Insight**

Small % of customers generate majority of revenue

In [64]:
clv = sales.groupby('CustomerID_y').agg({
    'SalesAmount': 'sum',
    'OrderID': 'nunique'
}).reset_index()

clv['AvgOrderValue'] = clv['SalesAmount'] / clv['OrderID']

clv['CLV'] = clv['AvgOrderValue'] * clv['OrderID']

fig = px.histogram(
    clv,
    x='CLV',
    nbins=50,
    title='💰 Customer Lifetime Value Distribution',

)

fig.show()

 # 🔁 Customer Retention Analysis

**Business Question**

Do customers come back or churn

**Insight**

Retention drops after first purchase → churn risk

In [65]:

first_purchase = sales.groupby('CustomerID_y')['OrderDate_y'].min().reset_index()
first_purchase.columns = ['CustomerID', 'FirstPurchase']

sales_ret = sales.merge(first_purchase, left_on='CustomerID_y', right_on='CustomerID')

sales_ret['Month'] = sales_ret['OrderDate_y'].dt.to_period('M')
sales_ret['CohortMonth'] = sales_ret['FirstPurchase'].dt.to_period('M')

sales_ret['CohortIndex'] = (sales_ret['Month'] - sales_ret['CohortMonth']).apply(lambda x: x.n)

cohort = sales_ret.pivot_table(
    index='CohortMonth',
    columns='CohortIndex',
    values='CustomerID',
    aggfunc='nunique'
)

cohort = cohort.divide(cohort.iloc[:,0], axis=0)

In [68]:
cohort.index = cohort.index.astype(str)

fig = px.imshow(
    cohort,
    title='🔁 Customer Retention Cohort Analysis'
)

fig.show()

## 🔍 Key Insights

- A small number of products contribute to the majority of revenue (Pareto Principle)  
- Sales show clear seasonal trends  
- Top customers generate a significant portion of total revenue  
- Discounts do not always increase sales  
- Certain shipping partners handle most orders efficiently  

## 💡 Business Recommendations

- Focus marketing efforts on top-performing products  
- Implement loyalty programs for high-value customers  
- Optimize discount strategies to avoid revenue loss  
- Strengthen partnerships with high-performing shipping providers  
- Increase inventory for high-demand categories  

## ✅ Conclusion

This analysis provides valuable insights into sales performance, customer behavior, and operational efficiency.

The findings can help businesses make data-driven decisions to improve revenue, optimize operations, and enhance customer satisfaction.